In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import ListedColormap
import os
from comut import comut
from comut import fileparsers
import palettable
import matplotlib

In [ ]:
#library and functions
def plot_categorical(comut, data, name, column, mapping = None):
    data = data.copy()
    data['category'] = name
    data['value'] = data[column]
    comut.add_categorical_data(data, name = name, mapping = mapping)

def plot_continuous(comut, data, name, column, mapping, cat_mapping, value_range):
    data = data.copy()
    data['category'] = name
    data['value'] = data[column]
    comut.add_continuous_data(data, name = name, mapping = mapping, cat_mapping = cat_mapping, value_range = value_range)

def plot_bar(comut, data, name, columns, mapping=None):
    data = data.copy()
    data = data[['sample']+columns]
    comut.add_bar_data(data, stacked = True, name = name, ylabel = name, mapping=mapping)
    
def plot_scatter(comut, data, name, columns, mapping=None, scatter_kwargs=None):
    data = data.copy()
    data = data[['sample']+columns]
    comut.add_scatter_data(data, stacked = True, name = name, ylabel = name, mapping=mapping, scatter_kwargs=scatter_kwargs)

In [ ]:

purp_7 = palettable.cartocolors.sequential.Purp_7.mpl_colormap
vivid_10 = palettable.cartocolors.qualitative.Vivid_10.mpl_colors
balance_6 = palettable.cmocean.diverging.Balance_6.mpl_colors


#smaller plot
from matplotlib import rcParams
custom_rcParams = {
    'font.family': 'Arial',
    'font.size': 13,
    'axes.labelsize': 13,
    'legend.fontsize':9,
    'xtick.labelsize': 5,
    'ytick.labelsize': 9
}

# update rcParams
rcParams.update(custom_rcParams)

data['sample'] = data['sample']
data["Group"] = data["Group"].astype("category")
data.Group = pd.Categorical(data.Group, 
                    categories=['EAC Brain Mets', 'PCAWG Primary EAC', 'Hartwig EAC Mets'],
                    ordered=True)
data.sort_values(by=['Group', 'sample'], ascending=[True,True], inplace=True)

comut1 = comut.CoMut()

plot_categorical(comut1, data, "Group",'Group', mapping={'PCAWG Primary EAC':'#f7ee91', 'Hartwig EAC Mets':'#ffd700', 'EAC Brain Mets':'#F3BF5A'})



####
import pandas as pd

genes = ["CDK6", "EGFR", "CCND1", "KRAS", "ERBB2"]

long_data = data.melt(
    id_vars="sample",
    value_vars=genes,
    var_name="category",
    value_name="value"
)

# Categorize CN
# Ensure numeric
long_data['value'] = long_data['value'].astype(float)

# Handle missing values
long_data['value'] = long_data['value'].fillna(0)

def categorize_cn(x):
    if x >= 8:
        return 'Focal Amp'
    elif x >= 4:
        return 'Amplified'
    elif x <= 0.5:
        return 'Deleted'
    else:
        return 'Neutral'

long_data['value'] = long_data['value'].apply(categorize_cn)


# Merge group information back
long_data = long_data.merge(
    data[['sample', 'Group']],
    on='sample',
    how='left'
)


# Create gene-level CN summary per Group
summary_gene = (
    long_data
    .assign(
        CN_status=lambda df: df['value'].map({
            'Focal Amp': 'Amplified',
            'Amplified': 'Amplified',
            'Neutral': 'Neutral'
        })
    )
    .dropna(subset=['CN_status'])  # removes Deleted
    .groupby(['Group', 'category', 'CN_status'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure columns exist even if one category is absent
for col in ['Amplified', 'Neutral']:
    if col not in summary_gene.columns:
        summary_gene[col] = 0

mapping = {
    'Focal Amp': '#800026',     # dark red
    'Amplified': '#FC4E2A',     # red
    'Deleted': '#4575B4',       # blue
    'Neutral': '#D9D9D9'        # grey
}

comut1.add_categorical_data(
    long_data[['sample', 'category', 'value']],
    "Copy Number",
    mapping=mapping
)





# Define your color mapping
sig_mapping = {
    'MutSig 17': '#01665e',
    'MutSig MMRd': '#66013c',
    'Other': '#f5f5f5'
}

# Call your plot_bar function to add the stacked bar
plot_bar(
    comut=comut1,           # your CoMut plot object
    data=data,                 # dataframe with 'sample' + MutSig columns
    name='Mutational signatures',    # label for the bar
    columns=['MutSig 17', 'MutSig MMRd', 'Other'],  # order of stacking
    mapping=sig_mapping
)


#groups


### plot


heights = {'SNV': 5}
hspace = 0.08
wspace = 0.05


comut1.plot_comut(figsize = (14,8), x_padding = 0.08, y_padding = 0.07, tri_padding = 0.08, heights=heights,
                hspace = hspace, 
                wspace= wspace)

comut1.add_unified_legend(ncol = 1)

# Save the plot as a PNG file
output_file = ".png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')  # Save with high resolution
print(f"Plot saved to {output_file}")



In [ ]:
#oncogene Percentage per group comparison plot

import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Load input file
# -----------------------------
input_file = "CN_summary_by_group_and_gene.csv"
df = pd.read_csv(input_file)

# -----------------------------
# Calculate percent amplified
# -----------------------------
df["Total"] = df["Amplified"] + df["Neutral"]
df["Percent_Amplified"] = (df["Amplified"] / df["Total"]) * 100

# -----------------------------
# Pivot for plotting
# -----------------------------
plot_data = df.pivot(
    index="category",
    columns="Group",
    values="Percent_Amplified"
)

# -----------------------------
# Reorder groups
# -----------------------------
group_order = ['PCAWG Primary EAC', 'Hartwig EAC Mets', 'EAC Brain Mets']
plot_data = plot_data[group_order]

# -----------------------------
# Set custom colors
# -----------------------------
group_colors = {
    'PCAWG Primary EAC': '#f7ee91',
    'Hartwig EAC Mets': '#f9bcc5',
    'EAC Brain Mets': '#F3BF5A'

}

# -----------------------------
# Ensure gene order (optional)
# -----------------------------
gene_order = ["ERBB2", "EGFR", "CCND1", "KRAS", "CDK6"]
plot_data = plot_data.reindex(gene_order)

# -----------------------------
# Generate grouped bar plot
# -----------------------------
ax = plot_data.plot(
    kind="bar",
    color=[group_colors[g] for g in plot_data.columns],
    edgecolor="black"
)

plt.ylabel("Amplification Frequency (%)")
plt.xlabel("Gene")
plt.title("Oncogene Amplification Frequency by Group")
plt.ylim(0, 100)
plt.xticks(rotation=45)
plt.legend(title="Group")
plt.tight_layout()

# Save figure
plt.savefig("Oncogene_Amplification_Frequency.png", dpi=300)

plt.show()

# -----------------------------
# Save frequency table
# -----------------------------
df.to_csv("Oncogene_Amplification_Frequency_Table.csv", index=False)



import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Load input file
# -----------------------------
input_file = "CN_summary_by_group_and_gene.csv"
df = pd.read_csv(input_file)

# -----------------------------
# Calculate percent amplified
# -----------------------------
df["Total"] = df["Amplified"] + df["Neutral"]
df["Percent_Amplified"] = (df["Amplified"] / df["Total"]) * 100

# -----------------------------
# Pivot for plotting
# -----------------------------
plot_data = df.pivot(
    index="category",
    columns="Group",
    values="Percent_Amplified"
)

# -----------------------------
# Rename groups (shorter legend labels)
# -----------------------------
rename_map = {
    'PCAWG Primary EAC': 'Primary EAC',
    'Hartwig EAC Mets': 'EAC Mets',
    'EAC Brain Mets': 'Brain Mets'
}

# Apply renaming to columns/index depending on structure
plot_data = plot_data.rename(columns=rename_map, index=rename_map)

# -----------------------------
# Reorder groups
# -----------------------------
group_order = ['Primary EAC', 'EAC Mets', 'Brain Mets']
plot_data = plot_data[group_order]

# -----------------------------
# Custom colors
# -----------------------------
group_colors = {
    'Primary EAC': '#f7ee91',
    'EAC Mets': '#f9bcc5',
    'Brain Mets': '#F3BF5A'
}

# -----------------------------
# Gene order
# -----------------------------
gene_order = ["ERBB2", "EGFR", "CCND1", "KRAS", "CDK6"]
plot_data = plot_data.reindex(gene_order)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(10, 6))
ax = plot_data.plot(
    kind="bar",
    color=[group_colors[g] for g in plot_data.columns],
    edgecolor="black",
    linewidth=1.2,
    width=0.8,
    figsize=(10,6)
)

# -----------------------------
# Aesthetic adjustments
# -----------------------------
ax.set_ylabel("Amplification Frequency (%)", fontsize=14, fontweight='bold')
ax.set_xlabel("Gene", fontsize=14, fontweight='bold')
ax.set_title("Oncogene Amplification Frequency by Group", fontsize=16, fontweight='bold')
ax.set_ylim(0, 45)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(fontsize=14, loc='upper right', frameon=False, edgecolor='black')

plt.tight_layout()

# Save figure
plt.savefig("Oncogene_Amplification_Frequency_Aesthetic.png", dpi=300)

plt.show()

# -----------------------------
# Save frequency table
# -----------------------------
df.to_csv("Oncogene_Amplification_Frequency_Table.csv", index=False)



##erbb2 only
# -----------------------------
# Gene order
# -----------------------------
gene_order = ["ERBB2"]
plot_data = plot_data.reindex(gene_order)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(3, 6))
ax = plot_data.plot(
    kind="bar",
    color=[group_colors[g] for g in plot_data.columns],
    edgecolor="black",
    linewidth=1.2,
    width=0.8,
    figsize=(3,4)
)

# -----------------------------
# Aesthetic adjustments
# -----------------------------
ax.set_ylabel("ERBB2 Amplification (%)", fontsize=16, fontweight='bold')
ax.set_xlabel("Gene", fontsize=10, fontweight='bold')
#ax.set_title("ERBB2 Amp Frequency by Group", fontsize=10, fontweight='bold')
ax.set_ylim(0, 45)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
plt.xticks(rotation=45, ha='right')
plt.legend(fontsize=10, loc='upper left', frameon=False, 
           #edgecolor='black'
          )

plt.tight_layout()

# Save figure
plt.savefig("ERBB2_Amplification_Frequency_Aesthetic.png", dpi=300)

plt.show()

# -----------------------------
# Save frequency table
# -----------------------------
df.to_csv("ERBB2_Amplification_Frequency_Table.csv", index=False)
